# Sanad سند — question answering over Saudi law

## Part 3: the answering model

Two open 7B models that handle Arabic are compared: **ALLaM-7B-Instruct** (SDAIA, trained from scratch on Arabic and English) and **Qwen2.5-7B-Instruct**. Both run in 4-bit.

1. Multiple choice on ArabLegalEval questions: closed book, with the top 3 retrieved passages, and with only the gold passage.
2. Grounded answers in Arabic with a citation such as [2], on the 79 Najiz questions and 150 test questions; and the same questions with the right passage removed, where the right answer is "not found".
3. The better model on dev is fine-tuned with QLoRA to answer briefly, cite, and say when the law is not in the context; then step 2 is repeated.

Retriever: BM25 + the fine-tuned retriever from Part 2 if it is attached as input, otherwise BM25 + multilingual-e5-base.

Code: https://github.com/readaloush/sanad

**Settings:** Internet on, GPU accelerator.

In [ ]:
!pip -q install sentence-transformers rank_bm25 peft bitsandbytes accelerate
!git clone -q https://github.com/readaloush/sanad.git
%cd sanad
import os, torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
RETRIEVER = 'intfloat/multilingual-e5-base'
for root, dirs, files in os.walk('/kaggle/input'):
    if 'modules.json' in files and 'retriever' in root:
        RETRIEVER = root
        break
os.environ['RETRIEVER'] = RETRIEVER
print('retriever:', RETRIEVER)

## 1–2. Both models before fine-tuning

In [ ]:
!python -m sanad.answer_eval --models ALLaM-AI/ALLaM-7B-Instruct-preview Qwen/Qwen2.5-7B-Instruct --retriever "$RETRIEVER" --data data --out results 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading|Fetching'

## 3. Fine-tune the better model (by dev accuracy with retrieved passages)

In [ ]:
import json
r = json.load(open('results/answers.json'))['models']
name = max(r, key=lambda m: r[m]['mcq_dev']['retrieved'])
BEST = {'ALLaM-7B-Instruct-preview': 'ALLaM-AI/ALLaM-7B-Instruct-preview', 'Qwen2.5-7B-Instruct': 'Qwen/Qwen2.5-7B-Instruct'}[name]
os.environ['BEST'] = BEST
print('fine-tuning', BEST)

In [ ]:
!python -m sanad.sft --model "$BEST" --retriever "$RETRIEVER" --data data --out results/lora 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading|Fetching'

In [ ]:
!python -m sanad.answer_eval --models "$BEST" --adapter results/lora --retriever "$RETRIEVER" --tag _lora --data data --out results 2>&1 | grep -v -E 'it/s\]|Warning|warn|B/s\]|^$|Loading|Fetching'

## Keep results and the adapter

In [ ]:
!mkdir -p /kaggle/working/out && cp results/answers*.json results/answer_samples*.json /kaggle/working/out/ && cp -r results/lora /kaggle/working/out/lora
%cd /kaggle/working
!rm -rf /kaggle/working/sanad && ls -la /kaggle/working/out /kaggle/working/out/lora